# Ask-SLM Runner (Thin)
# Minimal flow: initialize workspace, prepare mappings, enrich metadata, build merged dataset, and run downstream analyses.


In [1]:
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "managers").exists() else cwd.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

try:
    display  # type: ignore[name-defined]
except NameError:
    def display(obj):
        print(obj)

from managers.analysis_pipeline_manager import AnalysisPipelineManager
from managers.mapping_config_manager import MappingConfigManager


In [2]:
workspace = "ABX"
manager = AnalysisPipelineManager()

workspace_root = project_root / "Workspaces"
if not workspace_root.exists():
    raise FileNotFoundError(
        f"Cannot locate Workspaces directory at {workspace_root}. "
        "Run from repo root or Notebooks/."
    )

init_result = manager.initialize_workspace(
    workspace=workspace,
    workspace_root=workspace_root,
)
context = init_result.context
runner_logger = init_result.logger
print(f"Using workspace root: {workspace_root}")


[2026-04-23 16:21:59] Log file path: /Users/yarintamam/Code/Ask-SLM/Workspaces/ABX/Logs/run_20260423_162159_392916.log
Using workspace root: /Users/yarintamam/Code/Ask-SLM/Workspaces


In [3]:
inputs_result = manager.prepare_workspace_inputs(
    context=context,
    logger=runner_logger,
)
metadata_table = inputs_result.metadata_table
taxonomy_table = inputs_result.taxonomy_table

mapping_result = manager.refresh_mapping_templates(
    metadata_df=metadata_table,
    context=context,
    logger=runner_logger,
)
sra_processor = mapping_result.processor
project_ids = mapping_result.project_ids

display(mapping_result.as_dataframe())


,project_id,config_exists,config_path,template_path
0,PRJNA974858,True,/Users/yarintamam/Code/Ask-SLM/Workspaces/ABX/...,/Users/yarintamam/Code/Ask-SLM/Workspaces/ABX/...


In [4]:
# Keep mapping review explicit in the notebook flow.
MappingConfigManager(
    config_dir=context.config_dir,
    logger=runner_logger.log,
).run()


✅ Loaded 27 fields from Metalog schema
📋 Loaded 1 project(s)



In [5]:
enrichment_result = manager.run_enrichment_stage(
    metadata_df=metadata_table,
    processor=sra_processor,
    project_ids=project_ids,
    context=context,
    logger=runner_logger,
    assembly_db_dir=context.cache_dir,
    strict_missing_configs=False,
)
enriched_metadata = enrichment_result.enriched_metadata

envo_result = manager.enrich_envo_columns(
    metadata_df=enriched_metadata,
    context=context,
    logger=runner_logger,
)
enriched_metadata = envo_result.enriched_metadata

display(
    pd.DataFrame(
        {
            "metric": ["samples", "projects", "mapped_columns", "missing_mapping_projects", "envo_terms_resolved"],
            "value": [
                len(enriched_metadata),
                enriched_metadata["bioproject"].nunique(),
                len(enrichment_result.mapped_cols),
                len(enrichment_result.missing_mapping_projects),
                len(envo_result.envo_report_df),
            ],
        }
    )
)


[2026-04-23 16:22:07] Enrichment cache miss: rebuilt and saved to /Users/yarintamam/Code/Ask-SLM/Workspaces/ABX/results/_cache


,metric,value
0,samples,960
1,projects,1
2,mapped_columns,0
3,missing_mapping_projects,0
4,envo_terms_resolved,0


In [6]:
taxonomy_result = manager.run_taxonomy_stage(
    taxonomy_df=taxonomy_table,
    context=context,
    logger=runner_logger,
    assembly_db_dir=context.cache_dir,
    merged_dump_path=project_root / "Accessors/TaxonomyAccessor/merged.dmp",
    taxon_key_sep="||",
)

merge_result = manager.run_merge_stage(
    enriched_metadata_df=enriched_metadata,
    abundance_matrix=taxonomy_result.abundance_matrix,
    context=context,
    logger=runner_logger,
    assembly_db_dir=context.cache_dir,
)

abundance_metadata_matrix = merge_result.abundance_metadata_matrix
metadata_columns = merge_result.metadata_columns
taxon_lookup_df = taxonomy_result.taxon_lookup_df

display(abundance_metadata_matrix[metadata_columns].head())


[2026-04-23 16:22:21] Taxonomy cache miss: rebuilt taxonomy features in /Users/yarintamam/Code/Ask-SLM/Workspaces/ABX/results
[2026-04-23 16:22:55] Merge cache miss: rebuilt merged dataset in /Users/yarintamam/Code/Ask-SLM/Workspaces/ABX/results


,acc,assay_type,center_name,consent,experiment,sample_name,instrument,librarylayout,libraryselection,librarysource,...,ena_first_public_run,ena_last_update_run,sample_name_sam,datastore_filetype,datastore_provider,datastore_region,attributes,jattr,run_file_version,bioproject
0,SRR24705147,WGS,STANFORD,public,SRX20483864,EAC_50,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=11618174256}, {k=bytes, v=3730732...","{""bases"": 11618174256, ""bytes"": 3730732746, ""r...",1,PRJNA974858
1,SRR24705364,WGS,STANFORD,public,SRX20484319,EAQ_57,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=4701894376}, {k=bytes, v=15178353...","{""bases"": 4701894376, ""bytes"": 1517835301, ""ru...",1,PRJNA974858
2,SRR24705548,WGS,STANFORD,public,SRX20484135,EAP_63,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=6101445704}, {k=bytes, v=19618863...","{""bases"": 6101445704, ""bytes"": 1961886381, ""ru...",1,PRJNA974858
3,SRR24705617,WGS,STANFORD,public,SRX20484066,BAM_10,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=6404763040}, {k=bytes, v=20559767...","{""bases"": 6404763040, ""bytes"": 2055976750, ""ru...",1,PRJNA974858
4,SRR24705539,WGS,STANFORD,public,SRX20484144,EAP_75,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=17239632600}, {k=bytes, v=5449924...","{""bases"": 17239632600, ""bytes"": 5449924461, ""r...",1,PRJNA974858


In [ ]:
shared_tax_accessor = manager.build_or_load_tax_accessor(
    taxon_lookup_df=taxon_lookup_df,
    context=context,
    logger=runner_logger,
    assembly_db_dir=context.cache_dir,
    lineage_db_path=context.cache_dir / "taxonomy_lineage_db.json",
)


In [ ]:
manager.run_alpha_diversity(
    analysis_wide_df=abundance_metadata_matrix,
    taxon_lookup_df=taxon_lookup_df,
    metadata_cols=metadata_columns,
    tax_accessor=shared_tax_accessor,
    logger=runner_logger,
)


In [ ]:
manager.run_taxonomic_abundance(
    abundance_metadata_matrix=abundance_metadata_matrix,
    taxon_lookup_df=taxon_lookup_df,
    metadata_cols=metadata_columns,
    logger=runner_logger,
)
